In [3]:
from ingest import load_documents, build_index
from rag_helper import RAGHelper


In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

client = OpenAI(
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key= os.environ["OPENAI_API_KEY"]
)

In [5]:
documents = load_documents()
index = build_index(documents)

jarvis = RAGHelper(index, client)


In [8]:
answer , usage = jarvis.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)
print(usage)

The agent loop simply runs the model over and over until a turn comes back with **no function‑call entries**.  

In each iteration it:
1. Sends the current `messages` list to the model (with the tools attached).  
2. Appends the model’s output to `messages`.  
3. Scans the output for any items of type `function_call`.  
   * If it finds one, it runs the requested tool, adds the tool‑result back into `messages`, and sets a flag `has_function_calls = True`.  
   * If it finds no function calls, `has_function_calls` stays `False`.  

After processing the output the loop checks the flag:

```python
if has_function_calls == False:
    break
```

When the flag is false—that is, the model returned a normal message with no further tool calls—the `while True` loop exits. This is how the agent keeps calling the model repeatedly until the model decides it’s done.
7193
